# CliniBridge Demonstration Notebook

This notebook is designed for recording a walkthrough of the CliniBridge pipeline.

For each alert scenario it will:
- load the scenario input,
- run the orchestrator,
- show the path taken through the workflow,
- show which agents were triggered,
- summarize the audit log,
- print the final JSON log path and PDF path.

## 0. Optional Setup

If you need to install dependencies in the active notebook kernel, uncomment and run the next cell.

In [7]:
# !pip install -r requirements.txt ipykernel

## 1. Environment Setup

This cell finds the project root, loads `.env`, and verifies that the API key is available.

In [8]:
import json
import os
import sys
from glob import glob
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from IPython.display import Markdown, display

cwd = Path.cwd()
candidates = [cwd, cwd / "CliniBridge", cwd.parent, cwd.parent / "CliniBridge"]
BASE_DIR = None

for candidate in candidates:
    if (candidate / "Orchestrator").exists() and (candidate / "data").exists():
        BASE_DIR = candidate.resolve()
        break

if BASE_DIR is None:
    raise RuntimeError("Could not locate the CliniBridge project root.")

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

env_path = BASE_DIR / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"Environment loaded from: {env_path}")
else:
    print("Warning: .env file not found.")

api_key = os.getenv("OPENROUTER_API_KEY")
if api_key and "your_openrouter" not in api_key.lower():
    print("API key check: configured")
else:
    print("API key check: missing or placeholder value")

print(f"Project root: {BASE_DIR}")

Environment loaded from: C:\Users\wajee\PycharmProjects\CliniBridge\.env
API key check: configured
Project root: C:\Users\wajee\PycharmProjects\CliniBridge


## 2. Initialize the Orchestrator

This compiles the LangGraph workflow and generates `graph.png` if supported in the current environment.

In [9]:
from Orchestrator.orchestrator import Orchestrator

print("Initializing orchestrator...")
orchestrator = Orchestrator()
print("Orchestrator ready.")

Initializing orchestrator...
Graph saved to graph.png
Orchestrator ready.


## 3. Scenario Catalog

These are the five built-in alert demonstrations available in `data/alerts/`.

In [10]:
SCENARIOS = {
    1: "Missed Medication - BP spike after stopping Lisinopril",
    2: "False Alarm - High glucose due to planned dietary change",
    3: "Silent Deterioration - Weight gain trend in heart failure",
    4: "Incomplete Record - Transfer patient with sparse EHR",
    5: "Conflicting Data - Claimed adherence vs sub-therapeutic labs",
}

for number, title in SCENARIOS.items():
    print(f"Scenario {number}: {title}")

Scenario 1: Missed Medication - BP spike after stopping Lisinopril
Scenario 2: False Alarm - High glucose due to planned dietary change
Scenario 3: Silent Deterioration - Weight gain trend in heart failure
Scenario 4: Incomplete Record - Transfer patient with sparse EHR
Scenario 5: Conflicting Data - Claimed adherence vs sub-therapeutic labs


## 4. Demo Helpers

These helpers load scenarios, detect newly created artifacts in the patient `logs/` folder, summarize the workflow path, and print the most useful details for the demo.

In [11]:
def load_scenario(scenario_number: int) -> dict:
    path = BASE_DIR / "data" / "alerts" / f"scenario_{scenario_number}.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def patient_logs_dir(patient_id: str) -> Path:
    return BASE_DIR / "data" / "patients" / patient_id / "logs"


def snapshot_log_files(patient_id: str) -> dict:
    logs_dir = patient_logs_dir(patient_id)
    logs_dir.mkdir(parents=True, exist_ok=True)
    return {
        "json": {str(path.resolve()) for path in logs_dir.glob("*.json")},
        "pdf": {str(path.resolve()) for path in logs_dir.glob("*.pdf")},
    }


def detect_new_artifacts(patient_id: str, before: dict) -> dict:
    after = snapshot_log_files(patient_id)
    new_json = sorted(after["json"] - before["json"])
    new_pdf = sorted(after["pdf"] - before["pdf"])

    if not new_json:
        new_json = sorted(after["json"])
    if not new_pdf:
        new_pdf = sorted(after["pdf"])

    return {
        "json_log_path": new_json[-1] if new_json else None,
        "pdf_path": new_pdf[-1] if new_pdf else None,
        "logs_dir": str(patient_logs_dir(patient_id).resolve()),
    }


def infer_route_from_state(state: dict) -> list:
    route = ["safety_check"]

    if state.get("escalate"):
        route.append("emergency_report")
        return route

    triage_output = state.get("triage_output")
    if triage_output is not None or any(entry.get("node") == "triage" for entry in state.get("audit_log", [])):
        route.append("triage")

    severity = state.get("severity_class")
    if severity in {"CRITICAL", "HIGH"}:
        route.append("emergency_report")
        return route

    if state.get("ehr_findings") is not None or state.get("anamnesis_findings") is not None:
        route.append("parallel_retrieval")

    if state.get("clinical_brief") is not None:
        route.append("synthesis")

    return route


def triggered_agents_from_route(route: list) -> list:
    mapping = {
        "triage": "Triage Agent",
        "parallel_retrieval": "EHR Agent + Anamnesis Agent",
        "synthesis": "Synthesis Agent",
        "emergency_report": "Emergency Report Node (non-LLM)",
    }
    return [mapping[node] for node in route if node in mapping]


def summarize_audit_log(audit_log: list) -> None:
    print("Audit log trace:")
    for index, entry in enumerate(audit_log, start=1):
        node = entry.get("node", "unknown")
        print(f"  {index}. {node}")

        if node == "safety_check":
            print(f"     escalate={entry.get('escalate')} | reason={entry.get('reason')}")

        elif node == "triage":
            print(f"     severity_class={entry.get('severity_class')}")
            triage_output = entry.get("triage_output", {})
            ehr_queries = triage_output.get("ehr_queries", [])
            anamnesis_queries = triage_output.get("anamnesis_queries", [])
            print(f"     ehr_queries={len(ehr_queries)} | anamnesis_queries={len(anamnesis_queries)}")

        elif node == "parallel_retrieval":
            print(f"     ehr_results={len(entry.get('ehr_findings', []))} | anamnesis_results={len(entry.get('anamnesis_findings', []))}")

        elif node == "synthesis":
            brief = entry.get("clinical_brief", {})
            print(f"     final_brief_sections={len(brief) if isinstance(brief, dict) else 0}")

        elif node == "emergency_report":
            output = entry.get("output", {})
            print(f"     status={output.get('status')}")


def run_scenario(scenario_number: int) -> dict:
    alert = load_scenario(scenario_number)
    patient_id = alert["patient_id"]
    before = snapshot_log_files(patient_id)

    print("=" * 80)
    print(f"SCENARIO {scenario_number}: {alert.get('description')}")
    print("=" * 80)
    print(f"Alert file: {(BASE_DIR / 'data' / 'alerts' / f'scenario_{scenario_number}.json').resolve()}")
    print(f"Patient ID: {patient_id}")
    print(f"Reading: {alert.get('reading')} = {alert.get('value')} {alert.get('unit')}")
    print(f"Threshold: {alert.get('threshold')} | Alert level: {alert.get('alert_level')}")
    print(f"Timestamp: {alert.get('timestamp')}")
    print()
    print("Scenario payload:")
    pprint(alert)
    print()
    print("Running orchestrator...")

    state = orchestrator.run(alert)
    artifacts = detect_new_artifacts(patient_id, before)
    route = infer_route_from_state(state)
    triggered_agents = triggered_agents_from_route(route)

    print()
    print("Workflow path taken:")
    print(" -> ".join(route))
    print()
    print("Triggered agents / nodes:")
    for item in triggered_agents:
        print(f"- {item}")

    print()
    print(f"Severity class: {state.get('severity_class')}")
    print(f"Escalate flag after safety check: {state.get('escalate')}")

    triage_output = state.get("triage_output") or {}
    if triage_output:
        print(f"Triage EHR queries: {len(triage_output.get('ehr_queries', []))}")
        print(f"Triage anamnesis queries: {len(triage_output.get('anamnesis_queries', []))}")

    print()
    summarize_audit_log(state.get("audit_log", []))

    print()
    print("Output artifact paths:")
    print(f"- Logs directory: {artifacts['logs_dir']}")
    print(f"- JSON log: {artifacts['json_log_path']}")
    print(f"- PDF brief: {artifacts['pdf_path']}")

    errors = state.get("errors", [])
    if errors:
        print()
        print("Errors:")
        for err in errors:
            print(f"- {err}")

    print()
    print("Clinical brief preview:")
    print(json.dumps(state.get("clinical_brief"), indent=2, ensure_ascii=False))
    print()

    return {
        "scenario_number": scenario_number,
        "alert": alert,
        "state": state,
        "route": route,
        "artifacts": artifacts,
    }

## 5. Run All Scenarios in Sequence

Use this cell for a full end-to-end demo recording.

In [12]:
all_results = {}
for scenario_number in range(1, 6):
    all_results[scenario_number] = run_scenario(scenario_number)

SCENARIO 1: Missed Medication — BP spike after stopping Lisinopril
Alert file: C:\Users\wajee\PycharmProjects\CliniBridge\data\alerts\scenario_1.json
Patient ID: P001
Reading: blood_pressure = 178/108 mmHg
Threshold: 140/90 | Alert level: HIGH
Timestamp: 2026-05-14T09:31:00Z

Scenario payload:
{'alert_level': 'HIGH',
 'description': 'Missed Medication — BP spike after stopping Lisinopril',
 'device': 'Omron BP Monitor',
 'patient_id': 'P001',
 'reading': 'blood_pressure',
 'scenario_id': 'scenario_1',
 'threshold': '140/90',
 'timestamp': '2026-05-14T09:31:00Z',
 'unit': 'mmHg',
 'value': '178/108'}

Running orchestrator...


router: triage (severity=HIGH) → emergency_report
emergency_report_node: patient=P001



Workflow path taken:
safety_check -> triage -> emergency_report

Triggered agents / nodes:
- Triage Agent
- Emergency Report Node (non-LLM)

Severity class: HIGH
Escalate flag after safety check: False
Triage EHR queries: 5
Triage anamnesis queries: 4

Audit log trace:
  1. safety_check
     escalate=False | reason=None
  2. triage
     severity_class=HIGH
     ehr_queries=5 | anamnesis_queries=4
  3. emergency_report
     status=EMERGENCY ESCALATION

Output artifact paths:
- Logs directory: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P001\logs
- JSON log: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P001\logs\P001_20260623_134200.json
- PDF brief: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P001\logs\P001_20260623_134200.pdf

Clinical brief preview:
{
  "status": "EMERGENCY ESCALATION",
  "message": "Alert requires immediate human attention. Full AI synthesis skipped.",
  "action": "Contact clinician and patient NOW. Do not wait.",
  "patient_id"

safety_check: unknown reading type 'weight' — passing to triage



Workflow path taken:
safety_check -> triage -> parallel_retrieval -> synthesis

Triggered agents / nodes:
- Triage Agent
- EHR Agent + Anamnesis Agent
- Synthesis Agent

Severity class: MODERATE
Escalate flag after safety check: False
Triage EHR queries: 5
Triage anamnesis queries: 4

Audit log trace:
  1. safety_check
     escalate=False | reason=None
  2. triage
     severity_class=MODERATE
     ehr_queries=5 | anamnesis_queries=4
  3. parallel_retrieval
     ehr_results=5 | anamnesis_results=4
  4. synthesis
     final_brief_sections=6

Output artifact paths:
- Logs directory: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P002\logs
- JSON log: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P002\logs\P002_20260623_134308.json
- PDF brief: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P002\logs\P002_20260623_134308.pdf

Clinical brief preview:
{
  "alert_summary": {
    "trigger": "Blood glucose reading of 210 mg/dL detected by Dexcom G6 CGM, exceeding

## 6. Per-Scenario Cells

Use these if you want to record each scenario separately.

In [ ]:
scenario_1_result = run_scenario(1)

In [ ]:
scenario_2_result = run_scenario(2)

In [ ]:
scenario_3_result = run_scenario(3)

In [ ]:
scenario_4_result = run_scenario(4)

In [ ]:
scenario_5_result = run_scenario(5)

## 7. Compact Summary Table

After running the scenarios, this gives a concise summary of route and artifact paths.

In [16]:
summary_source = all_results if 'all_results' in globals() and all_results else {
    1: globals().get('scenario_1_result'),
    2: globals().get('scenario_2_result'),
    3: globals().get('scenario_3_result'),
    4: globals().get('scenario_4_result'),
    5: globals().get('scenario_5_result'),
}

for number, result in summary_source.items():
    if not result:
        continue

    print("-" * 80)
    print(f"Scenario {number}: {result['alert'].get('description')}")
    print(f"Route: {' -> '.join(result['route'])}")
    print(f"JSON log: {result['artifacts']['json_log_path']}")
    print(f"PDF brief: {result['artifacts']['pdf_path']}")

--------------------------------------------------------------------------------
Scenario 1: Missed Medication — BP spike after stopping Lisinopril
Route: safety_check -> triage -> emergency_report
JSON log: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P001\logs\P001_20260623_134200.json
PDF brief: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P001\logs\P001_20260623_134200.pdf
--------------------------------------------------------------------------------
Scenario 2: False Alarm — High glucose due to planned dietary change
Route: safety_check -> triage -> parallel_retrieval -> synthesis
JSON log: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P002\logs\P002_20260623_134308.json
PDF brief: C:\Users\wajee\PycharmProjects\CliniBridge\data\patients\P002\logs\P002_20260623_134308.pdf
--------------------------------------------------------------------------------
Scenario 3: Silent Deterioration — Gradual weight gain trend in heart failure patient
Route: s